# 02 -- Baseline Contact Model

**Contact Luck Prototype v0.1**

Trains the Version 0.1 baseline multinomial logistic regression on the cleaned, training-eligible contact events, using the time-based development design (train on 2021-2023, validate on 2024).

Prefers the full 2021-2024 development dataset (`cleaned_development_data.parquet`) when available, and falls back to the smaller one-week bootstrap sample (`cleaned_batted_balls.parquet`) -- which will show 0 training rows, since it only covers one week of 2024.

> **2025 is a protected, untouched final-test season. Never tune, iterate, or select features using 2025 results.** This notebook never loads 2025 data.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [ ]:
import pandas as pd

from mlb_luck_score.config import CLASS_ORDER, TRAIN_SEASONS, VALIDATION_SEASONS
from mlb_luck_score.models.train_contact_model import (
    evaluate_model,
    predict_proba_ordered,
    train_model,
    validate_probabilities,
)

pd.set_option("display.width", 120)

In [ ]:
from mlb_luck_score.config import DEVELOPMENT_SEASONS, PROCESSED_DATA_DIR

DEVELOPMENT_PATH = PROCESSED_DATA_DIR / "cleaned_development_data.parquet"
SAMPLE_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"

if DEVELOPMENT_PATH.exists():
    df = pd.read_parquet(DEVELOPMENT_PATH)
    print(f"Loaded the full development dataset: {len(df)} rows from {DEVELOPMENT_PATH}")
elif SAMPLE_PATH.exists():
    df = pd.read_parquet(SAMPLE_PATH)
    print(
        f"Full development dataset not found at {DEVELOPMENT_PATH}.\n"
        f"Falling back to the one-week bootstrap sample: {len(df)} rows from {SAMPLE_PATH}.\n"
        "Run `make download-development-data` then `make clean-development-data` for the "
        "full 2021-2024 dataset."
    )
else:
    df = None
    print(
        "No cleaned data found. Run one of:\n"
        "  make download-sample && make clean-data                     # small one-week sample\n"
        "  make download-development-data && make clean-development-data # full 2021-2024 dataset\n"
        "then re-run this notebook."
    )

In [ ]:
if df is not None:
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    print(f"{len(training_eligible)} of {len(df)} rows are training-eligible")
else:
    training_eligible = None

## Time-based split

Train seasons: configured in `mlb_luck_score.config.TRAIN_SEASONS`. Validation seasons: `mlb_luck_score.config.VALIDATION_SEASONS`. 2025 (`FINAL_TEST_SEASONS`) is never touched here.

In [ ]:
if training_eligible is not None:
    train_df = training_eligible[training_eligible["season"].isin(TRAIN_SEASONS)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
    print(f"Train seasons {TRAIN_SEASONS}: {len(train_df)} rows total")
    print(f"Validation seasons {VALIDATION_SEASONS}: {len(val_df)} rows total")
else:
    train_df = val_df = None
    print("Skipped -- no data loaded.")

### Training rows by season (2021-2023) and validation rows (2024)

In [ ]:
if train_df is not None:
    print("Training rows by season:")
    display(train_df.groupby("season").size().rename("training_rows").reindex(list(TRAIN_SEASONS), fill_value=0))
    print("\nValidation rows by season:")
    display(val_df.groupby("season").size().rename("validation_rows").reindex(list(VALIDATION_SEASONS), fill_value=0))
else:
    print("Skipped -- no data loaded.")

### Outcome counts by split

In [ ]:
if train_df is not None:
    train_counts = train_df["outcome_class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    val_counts = val_df["outcome_class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    outcome_by_split = pd.DataFrame({"train": train_counts, "validation": val_counts})
    display(outcome_by_split)
else:
    print("Skipped -- no data loaded.")

### Are all five outcome classes represented?

In [ ]:
if train_df is not None:
    train_present = set(train_df["outcome_class"].dropna().unique())
    val_present = set(val_df["outcome_class"].dropna().unique())
    train_missing = set(CLASS_ORDER) - train_present
    val_missing = set(CLASS_ORDER) - val_present

    if train_missing:
        print(f"Training split is MISSING class(es): {sorted(train_missing)}")
    else:
        print("Training split: all five outcome classes represented.")

    if val_missing:
        print(f"Validation split is MISSING class(es): {sorted(val_missing)}")
    else:
        print("Validation split: all five outcome classes represented.")
else:
    print("Skipped -- no data loaded.")

## Train the baseline pipeline

In [ ]:
if train_df is not None and len(train_df) > 0:
    trained = train_model(train_df)
    print("Numeric features:", trained.numeric_features)
    print("Categorical features:", trained.categorical_features)
else:
    trained = None
    print(
        "Skipped -- no training-eligible rows for the configured train seasons. "
        "If you're using the one-week sample, this is expected: run "
        "`make download-development-data` and `make clean-development-data` for "
        "2021-2023 training data."
    )

## Probability predictions

In [ ]:
if trained is not None and val_df is not None and len(val_df) > 0:
    feature_cols = trained.numeric_features + trained.categorical_features
    proba_df = predict_proba_ordered(trained, val_df[feature_cols].head(10))
    validate_probabilities(proba_df)
    display(proba_df)
else:
    print("Skipped -- no trained model or no validation rows available.")

## Core evaluation metrics (validation season)

In [ ]:
if trained is not None and val_df is not None and len(val_df) > 0:
    metrics = evaluate_model(trained, val_df)
    print("Sample count:", metrics["sample_count"])
    print("Multiclass log loss:", metrics["multiclass_log_loss"])
    print("Brier score by class:", metrics["brier_score_by_class"])
    print("Outcome class frequencies:", metrics["outcome_class_frequencies"])
    print("Feature missingness:", metrics["feature_missingness"])
    print("\nNo claim of strong predictive performance is made here -- these are "
          "raw diagnostic numbers, not a validated result.")
else:
    print("Skipped -- no trained model or no validation rows available.")